# Vérification du téléchargement Sentinel-2

Ce notebook vérifie que `download.copernicus.download_sentinel2` trouve la scène Sentinel-2 la plus récente (sous un seuil de nuages) via le catalogue STAC public d'Element84, découpe chaque bande sur l'emprise par lecture fenêtrée du COG distant, reprojette en EPSG:2154, et affiche un composite couleur sur une carte.

La fusion des bandes (composite, indices...) sera traitée dans une étape ultérieure ; ce notebook se contente d'un aperçu visuel rapide (composite RGB naturel) pour vérifier que les bandes téléchargées sont cohérentes entre elles.

In [ ]:
import sys
from pathlib import Path

# GeoDataEngine/ (racine du package core/) et son parent (racine du repo, contient data/)
project_root = Path.cwd().parent
repo_root = project_root.parent
sys.path.insert(0, str(project_root))

from core.config import load_entities
from download.copernicus import download_sentinel2

## 1. Entité testée (issue du CSV)

In [ ]:
csv_path = repo_root / "data" / "configuration" / "entites_exemple.csv"
entity = load_entities(csv_path)[0]
entity

## 2. Téléchargement de la dernière scène Sentinel-2 (toutes bandes)

In [ ]:
written = download_sentinel2(entity.bbox)
written

## 3. Aperçu visuel (composite RGB naturel)

Composite rapide rouge/vert/bleu à partir des bandes individuelles, juste pour vérifier visuellement la cohérence des données téléchargées (la vraie fusion/harmonisation des bandes est prévue dans une étape de traitement ultérieure).

In [ ]:
import numpy as np
import rioxarray

import leafmap.foliumap as leafmap

red = rioxarray.open_rasterio(written["red"]).squeeze()
green = rioxarray.open_rasterio(written["green"]).squeeze()
blue = rioxarray.open_rasterio(written["blue"]).squeeze()

rgb = np.stack([red.values, green.values, blue.values])
vmax = np.nanpercentile(rgb, 98)
rgb_norm = np.clip(rgb / vmax * 255, 0, 255).astype("uint8")

rgb_da = red.copy()
rgb_path = written["red"].with_name("composite_rgb_preview.tif")

import rasterio

profile = {
    "driver": "GTiff",
    "height": rgb_norm.shape[1],
    "width": rgb_norm.shape[2],
    "count": 3,
    "dtype": "uint8",
    "crs": red.rio.crs,
    "transform": red.rio.transform(),
}
with rasterio.open(rgb_path, "w", **profile) as dst:
    dst.write(rgb_norm)

m = leafmap.Map()
m.add_raster(str(rgb_path), layer_name="Sentinel-2 (composite RGB)")
m